In [2]:
import scanpy as sc
import pandas as pd
import numpy as np

In [3]:
main = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/'
atlas_in = main + 'embryo_mnn_bbknn_shiny.h5ad'

In [4]:
atlas = sc.read_h5ad(atlas_in)

/home/bt392/miniconda3/envs/env_multiome/lib/python3.9/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/bt392/miniconda3/envs/env_multiome/lib/python3.9/site-packages/anndata/compat/__init__.py:232: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/home/bt392/miniconda3/envs/env_multiome/lib/python3.9/site-packages/anndata/compat/__init__.py:232: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


In [5]:
blood_meta = pd.read_csv(main + 'blood/blood_meta.csv')

In [7]:
atlas

AnnData object with n_obs × n_vars = 430339 × 27669
    obs: 'cell', 'sample', 'stage', 'stage.mapped', 'stage.collapsed', 'stage.mapped.collapsed', 'somite_count', 'tube_name', 'tube_name_corrected', 'celltype', 'celltype.extended', 'celltype.descendant', 'celltype.descendant.somites', 'doub.density', 'cluster', 'cluster.sub', 'embryo_version', 'S_score', 'G2M_score', 'phase', 'louvain', 'leiden'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mgi_symbol'
    uns: 'draw_graph', 'leiden', 'louvain', 'neighbors', 'pca', 'umap'
    obsm: 'X_draw_graph_fa', 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [8]:
atlas = atlas[atlas.obs['cell'].isin(blood_meta['cell'].to_numpy())]

In [9]:
sc.pp.filter_genes(atlas, min_counts=100)

/home/bt392/miniconda3/envs/env_multiome/lib/python3.9/site-packages/scanpy/preprocessing/_simple.py:249: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['n_counts'] = number


In [10]:
atlas

AnnData object with n_obs × n_vars = 58170 × 14301
    obs: 'cell', 'sample', 'stage', 'stage.mapped', 'stage.collapsed', 'stage.mapped.collapsed', 'somite_count', 'tube_name', 'tube_name_corrected', 'celltype', 'celltype.extended', 'celltype.descendant', 'celltype.descendant.somites', 'doub.density', 'cluster', 'cluster.sub', 'embryo_version', 'S_score', 'G2M_score', 'phase', 'louvain', 'leiden'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mgi_symbol', 'n_counts'
    uns: 'draw_graph', 'leiden', 'louvain', 'neighbors', 'pca', 'umap'
    obsm: 'X_draw_graph_fa', 'X_pca', 'X_tsne', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [11]:
test = np.asmatrix(atlas.X)

In [12]:
test[1:10, 1:10]

matrix([[1.4854922 , 2.2017136 , 0.        , 1.4854922 , 0.        ,
         0.        , 0.        , 1.4854922 , 0.        ],
        [2.9644275 , 0.        , 0.        , 0.97978044, 0.        ,
         0.        , 0.        , 0.        , 0.        ],
        [3.0595727 , 0.        , 0.        , 2.7802694 , 0.        ,
         0.        , 0.        , 0.        , 0.        ],
        [1.710209  , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        ],
        [0.        , 1.2801199 , 0.        , 1.2801199 , 1.2801199 ,
         0.        , 0.        , 0.        , 0.        ],
        [1.8865787 , 0.        , 0.        , 1.8865787 , 0.        ,
         0.92538506, 0.        , 0.        , 0.        ],
        [2.303833  , 0.        , 0.        , 2.303833  , 1.2094905 ,
         0.        , 0.        , 0.        , 0.        ],
        [2.248504  , 0.        , 0.        , 0.        , 0.        ,
         0.        , 0.        , 0.       

In [13]:
import rpy2.robjects as ro
import rpy2.robjects.numpy2ri
rpy2.robjects.numpy2ri.activate()

nr,nc = test.shape
Br = ro.r.matrix(test, nrow=nr, ncol=nc)

ro.r.assign("B", Br)

array([[0.        , 1.19030309, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.48549223, 2.20171356, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 2.96442747, 0.        , ..., 0.        , 0.97978044,
        0.        ],
       ...,
       [0.        , 2.09319901, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.9275347 , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.785743  , 1.785743  , ..., 0.        , 0.        ,
        0.        ]])

In [ ]:
ro.r('as.matrix(B[1:10, 1:10])')

In [26]:
import anndata2ri
from rpy2.robjects import r
anndata2ri.activate()
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [ ]:
%%R -i atlas

atlas
saveRDS('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/blood/atlas_sce.rds')

In [ ]:
%%R

In [49]:
cells = atlas.obs['cell']

In [65]:
umap = pd.DataFrame(atlas.obsm['X_umap'])
umap = umap.set_index(cells)

In [67]:
umap.columns = ['umapX', 'umapY']

In [69]:
umap.to_csv(main + 'umap.csv')